<a href="https://colab.research.google.com/github/DouglasLeite77/mecaniQA-MACAPA/blob/main/metricas_timeseries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import os

# Verifica se estamos no Colab e pede o arquivo se não existir
if not os.path.exists('./mecaniqa_dataset.xlsx'):
    try:
        from google.colab import files
        print("Faça o upload da planilha mecaniqa_dataset.xlsx abaixo:")
        uploaded = files.upload()
    except ImportError:
        pass # Se não estiver no Colab, ignora

# 1. CARREGAMENTO DOS DADOS REAIS
try:
    # Lê o arquivo que acabou de ser carregado
    df = pd.read_excel('mecaniqa_dataset.xlsx')
    df['Data'] = pd.to_datetime(df['Data'])
    df = df.sort_values('Data').set_index('Data')

    # Isola a série de manutenções
    serie_manutencoes = df.select_dtypes('number').iloc[:, 0].astype(float)
    print(f"Base de dados carregada com sucesso! Total de dias: {len(serie_manutencoes)}")

except Exception as e:
    print(f"Erro ao ler a base de dados: {e}. Verifique se o nome do arquivo está exato.")

# 2. FEATURE ENGINEERING: RECRIANDO OS BASELINES (Lags e Janelas Rolantes)
pred_naive = serie_manutencoes.shift(1)
pred_mm7 = serie_manutencoes.shift(1).rolling(window=7).mean()

Faça o upload da planilha mecaniqa_dataset.xlsx abaixo:


Saving mecaniqa_dataset.xlsx to mecaniqa_dataset.xlsx
Base de dados carregada com sucesso! Total de dias: 731


In [4]:
# 3. VALIDAÇÃO CRUZADA TEMPORAL (TimeSeriesSplit)
tscv = TimeSeriesSplit(n_splits=5)

# Listas para armazenar os resultados de cada janela (fold)
resultados_naive = {'mae': [], 'rmse': [], 'mape': []}
resultados_mm7 = {'mae': [], 'rmse': [], 'mape': []}

# Iterando cronologicamente no tempo
for train_index, test_index in tscv.split(serie_manutencoes):
    # Separando a fatia Real (y_true)
    y_test = serie_manutencoes.iloc[test_index]

    # Separando a fatia de Previsões (y_pred)
    y_pred_naive = pred_naive.iloc[test_index]
    y_pred_mm7 = pred_mm7.iloc[test_index]

    # Limpeza rápida de NaNs (originados pelos shifts iniciais)
    valid_idx_naive = ~np.isnan(y_pred_naive) & ~np.isnan(y_test)
    valid_idx_mm7 = ~np.isnan(y_pred_mm7) & ~np.isnan(y_test)

    # Calculando as métricas para a janela atual (NAIVE)
    if valid_idx_naive.sum() > 0:
        resultados_naive['mae'].append(mean_absolute_error(y_test[valid_idx_naive], y_pred_naive[valid_idx_naive]))
        resultados_naive['rmse'].append(np.sqrt(mean_squared_error(y_test[valid_idx_naive], y_pred_naive[valid_idx_naive])))
        resultados_naive['mape'].append(mean_absolute_percentage_error(y_test[valid_idx_naive], y_pred_naive[valid_idx_naive]) * 100)

    # Calculando as métricas para a janela atual (Média Móvel 7 dias)
    if valid_idx_mm7.sum() > 0:
        resultados_mm7['mae'].append(mean_absolute_error(y_test[valid_idx_mm7], y_pred_mm7[valid_idx_mm7]))
        resultados_mm7['rmse'].append(np.sqrt(mean_squared_error(y_test[valid_idx_mm7], y_pred_mm7[valid_idx_mm7])))
        resultados_mm7['mape'].append(mean_absolute_percentage_error(y_test[valid_idx_mm7], y_pred_mm7[valid_idx_mm7]) * 100)

# 4. OUTPUT FINAL (Critério de Aceite)
print("Resultados do Baseline (NAIVE) - MAE: {:.2f}, RMSE: {:.2f}, MAPE: {:.2f}%".format(
    np.mean(resultados_naive['mae']),
    np.mean(resultados_naive['rmse']),
    np.mean(resultados_naive['mape'])
))

print("Resultados do Baseline (MÉDIA MÓVEL 7) - MAE: {:.2f}, RMSE: {:.2f}, MAPE: {:.2f}%".format(
    np.mean(resultados_mm7['mae']),
    np.mean(resultados_mm7['rmse']),
    np.mean(resultados_mm7['mape'])
))

Resultados do Baseline (NAIVE) - MAE: 6.65, RMSE: 8.92, MAPE: 5210776428362590.00%
Resultados do Baseline (MÉDIA MÓVEL 7) - MAE: 7.36, RMSE: 8.09, MAPE: 9996183352369024.00%
